In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/kushal0418/synthetic-data-for-ess/synthetic_sample_ess.csv


In [2]:
df = pd.read_csv('/kaggle/input/datasets/kushal0418/synthetic-data-for-ess/synthetic_sample_ess.csv')

In [3]:
df

,X1,X2,X3,Cat1,Cat2,target_code,target
0,0.176133,1.727698,-1.319274,A,Yes,0,Category_1
1,-0.485588,0.845374,0.470220,C,Yes,0,Category_1
2,-0.450979,-1.266292,-1.306431,C,Yes,0,Category_1
3,-0.552348,-0.282998,-0.284167,C,Yes,1,Category_2
4,1.654038,0.546707,1.175560,B,No,3,Category_4
...,...,...,...,...,...,...,...
2995,-0.074525,0.353667,-0.513776,A,No,2,Category_3
2996,-0.085477,-0.774381,0.939707,A,No,3,Category_4
2997,-0.476895,-0.521033,-0.694200,B,Yes,3,Category_4
2998,1.862708,-1.513831,0.933014,C,No,3,Category_4


# ML Model Pipeline

In [4]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import warnings
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from statsmodels.miscmodels.ordinal_model import OrderedModel
from xgboost import XGBClassifier

In [5]:
# ============================================================
# 2. MAIN PIPELINE
# ============================================================

def run_classification_pipeline(
    dataset,
    target_name,
    numerical_covariates_name,
    categorical_covariates_name,
    test_prop,
    start_seed,
    end_seed
):
    """
    Fits multiple classification models across a range of random seeds.

    Parameters
    ----------
    dataset : pandas.DataFrame
        Input dataframe.

    target_name : str
        Name of the target column.

    numerical_covariates_name : list of str
        Names of numerical predictor columns.

    categorical_covariates_name : list of str
        Names of categorical predictor columns.

    test_prop : float
        Proportion of observations assigned to the test set.
        Example: 0.20 means 20% test data.

    start_seed : int
        First random seed, inclusive.

    end_seed : int
        Final random seed, inclusive.

    Returns
    -------
    pandas.DataFrame
        Flat dataframe containing one row for every
        random_seed × model × data combination.

        Output columns:
        random_seed, model, data, n_samples,
        accuracy, precision, recall, f1_score
    """

    # --------------------------------------------------------
    # 2.1 Validate the inputs
    # --------------------------------------------------------

    if not isinstance(dataset, pd.DataFrame):
        raise TypeError("dataset must be a pandas DataFrame.")

    if target_name not in dataset.columns:
        raise ValueError(
            f"Target column '{target_name}' was not found in the dataset."
        )

    if not isinstance(numerical_covariates_name, (list, tuple)):
        raise TypeError(
            "numerical_covariates_name must be a list or tuple."
        )

    if not isinstance(categorical_covariates_name, (list, tuple)):
        raise TypeError(
            "categorical_covariates_name must be a list or tuple."
        )

    numerical_covariates_name = list(numerical_covariates_name)
    categorical_covariates_name = list(categorical_covariates_name)

    predictor_names = (
        numerical_covariates_name +
        categorical_covariates_name
    )

    if len(predictor_names) == 0:
        raise ValueError(
            "At least one numerical or categorical covariate is required."
        )

    missing_columns = [
        column
        for column in predictor_names
        if column not in dataset.columns
    ]

    if missing_columns:
        raise ValueError(
            f"The following predictor columns were not found: "
            f"{missing_columns}"
        )

    duplicate_columns = set(
        numerical_covariates_name
    ).intersection(categorical_covariates_name)

    if duplicate_columns:
        raise ValueError(
            "The following columns are present in both numerical and "
            f"categorical covariates: {sorted(duplicate_columns)}"
        )

    if not 0 < test_prop < 1:
        raise ValueError("test_prop must be between 0 and 1.")

    if start_seed > end_seed:
        raise ValueError(
            "start_seed must be less than or equal to end_seed."
        )

    # --------------------------------------------------------
    # 2.2 Select required columns and remove missing targets
    # --------------------------------------------------------

    model_data = dataset[
        predictor_names + [target_name]
    ].copy()

    missing_target_count = model_data[target_name].isna().sum()

    if missing_target_count > 0:
        warnings.warn(
            f"{missing_target_count} rows with missing target values "
            "were removed."
        )

        model_data = model_data.dropna(
            subset=[target_name]
        ).reset_index(drop=True)

    if model_data.empty:
        raise ValueError(
            "No observations remain after removing missing target values."
        )

    # --------------------------------------------------------
    # 2.3 Encode the target
    # --------------------------------------------------------
    # LabelEncoder creates integer labels:
    # 0, 1, 2, ..., K-1
    #
    # If the target represents ordinal categories, its ordering must
    # be meaningful. Ordered pandas categoricals are handled separately.

    target_series = model_data[target_name]

    if isinstance(target_series.dtype, pd.CategoricalDtype):

        if target_series.cat.ordered:
            target_classes = list(
                target_series.cat.categories
            )

            target_mapping = {
                category: index
                for index, category in enumerate(target_classes)
            }

            y = target_series.map(target_mapping).astype(int).to_numpy()

        else:
            target_encoder = LabelEncoder()

            y = target_encoder.fit_transform(
                target_series.astype(str)
            )

            target_classes = list(target_encoder.classes_)

            warnings.warn(
                "The target is categorical but is not marked as ordered. "
                "Ordinal Logistic Regression will use the order produced "
                "by LabelEncoder. Mark the target as an ordered pandas "
                "Categorical before calling the function if a specific "
                "ordinal order is required."
            )

    else:
        target_encoder = LabelEncoder()

        y = target_encoder.fit_transform(
            target_series
        )

        target_classes = list(target_encoder.classes_)

    n_classes = len(target_classes)

    if n_classes < 2:
        raise ValueError(
            "The target must contain at least two different classes."
        )

    # --------------------------------------------------------
    # 2.4 Predictor dataframe
    # --------------------------------------------------------

    X = model_data[predictor_names].copy()

    # Store class mapping for reference
    class_mapping = {
        original_class: encoded_class
        for encoded_class, original_class
        in enumerate(target_classes)
    }

    print("Target class mapping:")

    for original_class, encoded_class in class_mapping.items():
        print(f"  {original_class} -> {encoded_class}")

    # --------------------------------------------------------
    # 2.5 Numerical preprocessing
    # --------------------------------------------------------
    # Median imputation is fitted using the training set.
    # StandardScaler is applied only to numerical columns.

    numerical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]
    )

    # --------------------------------------------------------
    # 2.6 Categorical preprocessing
    # --------------------------------------------------------
    # Most-frequent imputation and one-hot encoding are fitted
    # using the training set.
    #
    # drop="first" avoids a complete set of redundant dummy
    # variables, which is particularly important for OrderedModel.

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent")
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    drop="first",
                    sparse_output=False
                )
            )
        ]
    )

    transformers = []

    if numerical_covariates_name:
        transformers.append(
            (
                "numerical",
                numerical_pipeline,
                numerical_covariates_name
            )
        )

    if categorical_covariates_name:
        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_covariates_name
            )
        )

    preprocessing_pipeline = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0
    )

    # --------------------------------------------------------
    # 2.7 Storage for the final flat output
    # --------------------------------------------------------

    all_results = []

    # --------------------------------------------------------
    # 2.8 Metric calculation function
    # --------------------------------------------------------

    def calculate_metrics(
        random_seed,
        model_name,
        data_name,
        y_true,
        y_pred
    ):
        """
        Calculate weighted classification metrics.
        """

        return {
            "random_seed": random_seed,
            "model": model_name,
            "data": data_name,
            "n_samples": len(y_true),

            "accuracy": accuracy_score(
                y_true,
                y_pred
            ),

            "precision": precision_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0
            ),

            "recall": recall_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0
            ),

            "f1_score": f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0
            )
        }

    # ========================================================
    # 3. RANDOM-SEED LOOP
    # ========================================================

    for seed in range(start_seed, end_seed + 1):

        print("\n" + "=" * 70)
        print(f"Processing random seed: {seed}")
        print("=" * 70)

        # ----------------------------------------------------
        # 3.1 Stratified train-test split
        # ----------------------------------------------------

        try:
            X_train, X_test, y_train, y_test = train_test_split(
                X,
                y,
                test_size=test_prop,
                random_state=seed,
                stratify=y,
                shuffle=True
            )

        except ValueError as error:
            raise ValueError(
                f"Stratified train-test split failed for seed {seed}. "
                "Check whether every target class contains enough "
                f"observations. Original error: {error}"
            ) from error

        print(f"Training observations: {len(X_train)}")
        print(f"Testing observations:  {len(X_test)}")

        # ----------------------------------------------------
        # 3.2 Fit preprocessing only on the training data
        # ----------------------------------------------------

        seed_preprocessor = clone(preprocessing_pipeline)

        X_train_processed = seed_preprocessor.fit_transform(
            X_train
        )

        X_test_processed = seed_preprocessor.transform(
            X_test
        )

        # Explicit conversion for model compatibility
        X_train_processed = np.asarray(
            X_train_processed,
            dtype=np.float64
        )

        X_test_processed = np.asarray(
            X_test_processed,
            dtype=np.float64
        )

        # ----------------------------------------------------
        # 3.3 Define non-ordinal models
        # ----------------------------------------------------

        models = {
            "Multinomial Logistic Regression":
                LogisticRegression(
                    solver="lbfgs",
                    max_iter=5000,
                    random_state=seed
                ),

            "Decision Tree":
                DecisionTreeClassifier(
                    random_state=seed
                ),

            "Random Forest":
                RandomForestClassifier(
                    n_estimators=500,
                    random_state=seed,
                    n_jobs=-1
                ),

            "Gradient Boosting":
                GradientBoostingClassifier(
                    random_state=seed
                ),

            "XGBoost":
                XGBClassifier(
                    objective=(
                        "binary:logistic"
                        if n_classes == 2
                        else "multi:softprob"
                    ),
                    num_class=(
                        n_classes
                        if n_classes > 2
                        else None
                    ),
                    n_estimators=300,
                    learning_rate=0.05,
                    max_depth=6,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    random_state=seed,
                    n_jobs=-1,
                    eval_metric=(
                        "logloss"
                        if n_classes == 2
                        else "mlogloss"
                    )
                )
        }

        # ----------------------------------------------------
        # 3.4 Fit and evaluate the standard classifiers
        # ----------------------------------------------------

        for model_name, model in models.items():

            print(f"\nFitting: {model_name}")

            try:
                model.fit(
                    X_train_processed,
                    y_train
                )

                train_predictions = model.predict(
                    X_train_processed
                )

                test_predictions = model.predict(
                    X_test_processed
                )

                train_predictions = np.asarray(
                    train_predictions
                ).astype(int)

                test_predictions = np.asarray(
                    test_predictions
                ).astype(int)

                all_results.append(
                    calculate_metrics(
                        random_seed=seed,
                        model_name=model_name,
                        data_name="train",
                        y_true=y_train,
                        y_pred=train_predictions
                    )
                )

                all_results.append(
                    calculate_metrics(
                        random_seed=seed,
                        model_name=model_name,
                        data_name="test",
                        y_true=y_test,
                        y_pred=test_predictions
                    )
                )

                print(f"Completed: {model_name}")

            except Exception as error:
                warnings.warn(
                    f"{model_name} failed for seed {seed}. "
                    f"Error: {error}"
                )

                # Preserve the seed × model × data structure
                for data_name, sample_size in [
                    ("train", len(y_train)),
                    ("test", len(y_test))
                ]:
                    all_results.append(
                        {
                            "random_seed": seed,
                            "model": model_name,
                            "data": data_name,
                            "n_samples": sample_size,
                            "accuracy": np.nan,
                            "precision": np.nan,
                            "recall": np.nan,
                            "f1_score": np.nan
                        }
                    )

        # ----------------------------------------------------
        # 3.5 Fit Ordinal Logistic Regression
        # ----------------------------------------------------

        ordinal_model_name = "Ordinal Logistic Regression"

        print(f"\nFitting: {ordinal_model_name}")

        try:
            # Remove columns with effectively zero variance.
            # Constant columns can cause identification problems
            # in statsmodels OrderedModel.

            feature_variance = np.var(
                X_train_processed,
                axis=0
            )

            non_constant_mask = feature_variance > 1e-12

            if not np.any(non_constant_mask):
                raise ValueError(
                    "All processed predictors have zero variance."
                )

            X_train_ordinal = X_train_processed[
                :,
                non_constant_mask
            ]

            X_test_ordinal = X_test_processed[
                :,
                non_constant_mask
            ]

            # OrderedModel automatically estimates thresholds.
            # Do not manually add an intercept/constant.

            ordinal_model = OrderedModel(
                endog=y_train,
                exog=X_train_ordinal,
                distr="logit"
            )

            ordinal_result = ordinal_model.fit(
                method="bfgs",
                maxiter=1000,
                disp=False
            )

            # Predicted class probabilities
            train_ordinal_probabilities = ordinal_result.model.predict(
                ordinal_result.params,
                exog=X_train_ordinal
            )

            test_ordinal_probabilities = ordinal_result.model.predict(
                ordinal_result.params,
                exog=X_test_ordinal
            )

            # Select class with highest probability
            train_ordinal_predictions = np.argmax(
                np.asarray(train_ordinal_probabilities),
                axis=1
            )

            test_ordinal_predictions = np.argmax(
                np.asarray(test_ordinal_probabilities),
                axis=1
            )

            all_results.append(
                calculate_metrics(
                    random_seed=seed,
                    model_name=ordinal_model_name,
                    data_name="train",
                    y_true=y_train,
                    y_pred=train_ordinal_predictions
                )
            )

            all_results.append(
                calculate_metrics(
                    random_seed=seed,
                    model_name=ordinal_model_name,
                    data_name="test",
                    y_true=y_test,
                    y_pred=test_ordinal_predictions
                )
            )

            print(f"Completed: {ordinal_model_name}")

        except Exception as error:
            warnings.warn(
                f"{ordinal_model_name} failed for seed {seed}. "
                f"Error: {error}"
            )

            for data_name, sample_size in [
                ("train", len(y_train)),
                ("test", len(y_test))
            ]:
                all_results.append(
                    {
                        "random_seed": seed,
                        "model": ordinal_model_name,
                        "data": data_name,
                        "n_samples": sample_size,
                        "accuracy": np.nan,
                        "precision": np.nan,
                        "recall": np.nan,
                        "f1_score": np.nan
                    }
                )

    # ========================================================
    # 4. CREATE FINAL FLAT DATAFRAME
    # ========================================================

    final_results = pd.DataFrame(all_results)

    model_order = [
        "Multinomial Logistic Regression",
        "Ordinal Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting",
        "XGBoost"
    ]

    data_order = ["train", "test"]

    final_results["model"] = pd.Categorical(
        final_results["model"],
        categories=model_order,
        ordered=True
    )

    final_results["data"] = pd.Categorical(
        final_results["data"],
        categories=data_order,
        ordered=True
    )

    final_results = (
        final_results
        .sort_values(
            by=["random_seed", "model", "data"]
        )
        .reset_index(drop=True)
    )

    # Convert categoricals back to ordinary strings
    final_results["model"] = final_results["model"].astype(str)
    final_results["data"] = final_results["data"].astype(str)

    final_results = final_results[
        [
            "random_seed",
            "model",
            "data",
            "n_samples",
            "accuracy",
            "precision",
            "recall",
            "f1_score"
        ]
    ]

    print("\n" + "=" * 70)
    print("Pipeline completed successfully.")
    print("=" * 70)

    return final_results

In [6]:
df.columns

Index(['X1', 'X2', 'X3', 'Cat1', 'Cat2', 'target_code', 'target'], dtype='object')

In [7]:
model_outputs = run_classification_pipeline(
    dataset = df,
    target_name = 'target',
    numerical_covariates_name = ['X1', 'X2', 'X3'],
    categorical_covariates_name = ['Cat1', 'Cat2'],
    test_prop = 0.2,
    start_seed = 40,
    end_seed = 61
)

Target class mapping:
  Category_1 -> 0
  Category_2 -> 1
  Category_3 -> 2
  Category_4 -> 3

Processing random seed: 40
Training observations: 2400
Testing observations:  600

Fitting: Multinomial Logistic Regression
Completed: Multinomial Logistic Regression

Fitting: Decision Tree
Completed: Decision Tree

Fitting: Random Forest
Completed: Random Forest

Fitting: Gradient Boosting
Completed: Gradient Boosting

Fitting: XGBoost
Completed: XGBoost

Fitting: Ordinal Logistic Regression
Completed: Ordinal Logistic Regression

Processing random seed: 41
Training observations: 2400
Testing observations:  600

Fitting: Multinomial Logistic Regression
Completed: Multinomial Logistic Regression

Fitting: Decision Tree
Completed: Decision Tree

Fitting: Random Forest
Completed: Random Forest

Fitting: Gradient Boosting
Completed: Gradient Boosting

Fitting: XGBoost
Completed: XGBoost

Fitting: Ordinal Logistic Regression
Completed: Ordinal Logistic Regression

Processing random seed: 42
Trai

In [8]:
model_outputs.to_csv('ess_expt_model_output_ml.csv',index = False)